# Stereo-seq (FF footpad tumors): exploratory analyses (not shown as figure panels)
- T-cell distribution
- Screen for melanoma signatures (AUCell scores stored in obs.aucell_*) that are higher in the Running group and decrease with distance from the tumor border
- Gene-level heatmaps for those signatures

In [ ]:
import os
import re
import importlib.metadata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from scipy import stats

from spatial_utils import *

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')
os.makedirs('./figures', exist_ok=True)

for package in ['stereopy', 'numpy', 'pandas', 'matplotlib', 'scanpy', 'anndata', 'squidpy', 'decoupler']:
    try:
        print(f"{package} version: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed")

# Annotated bin30 objects: <GEO library name>_bin30_annotated.h5ad
SPATIAL_DIR = '../data/stereoseq_annotated'
sample_list = SAMPLE_LIST

In [ ]:
adataL = load_margin_lognorm(SPATIAL_DIR, sample_list)
lognorm4 = load_scored(SPATIAL_DIR, sample_list)

## T cells

In [ ]:
layers_to_view = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um', '500um~550um', '550um~600um', '600um~650um', '650um~700um', '700um~750um', '750um~800um', '800um~850um', '850um~900um', '900um~950um', '950um~1000um', '1000um~1050um', '1050um~1100um', '1100um~1150um', '1150um~1200um', '1200um~1250um', '1250um~1300um', '1300um~1350um', '1350um~1400um', '1400um~1450um', '1450um~1500um']

In [ ]:
sc.set_figure_params(dpi=300, color_map = 'OrRd', facecolor=None, transparent=True)

In [ ]:
for sample in sample_list:
    if sample not in ['FF_Mattress_1', 'FF_Running_3', 'FF_YAP5SA_2']:
        bdata = adataL[sample]
        bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
        sc.pl.spatial(bdata, color = 'cell_type', spot_size=30, frameon=False, title = f'{sample}', groups = ['T'])

In [ ]:
proportion_dict = {}
for sample in sample_list:
    df = pd.crosstab(adataL[sample].obs['cell_type_level2'], adataL[sample].obs['tumor_layer'])
    df = df.div(df.sum(axis=0), axis = 1)
    proportion_dict[sample] = df
    del df

In [ ]:
plot_cell_type_proportions_line(proportion_dict, cell_types='T')

## Gene sets
AUCell scores (including REACTOME_INTERLEUKIN_1_FAMILY_SIGNALING) are already stored in `obs.aucell_*`; gene sets are only needed for the gene-level heatmaps.

In [ ]:
import json

# Gene sets used in the original analysis (MSigDB Hallmark / curated (m2) / ontology, mouse),
# exported to ../data/genesets_used.json
with open('../data/genesets_used.json', 'r') as f:
    new_dict = json.load(f)

In [ ]:
# genes present in FF_YAP5SA_2 (the last sample of the preceding loop in the original notebook)
YAP_sig = 'NDRG1,STMN1,TSPAN3,TGM2,DUSP1,THBS1,THBS1,THBS1,ITGB5,DAB2,DAB2,DAB2,CCN1,TOP2A,PHGDH,FSCN1,MARCKS,SGK1,HEXB,BIRC5,GAS6,TK1,CRIM1,SERPINE1,AXL,ITGB2,CDC20,AMOTL2,ETV5,GGH,SCHIP1,CDKN2C,FGF2,FGF2,EMP2,ANKRD1,HMMR,FSTL1,DUT,DDAH1,CCN2,CENPF,GADD45B,SLIT2,TGFB2,TGFB2,PMP22,DAB2,DLC1,CCN1,GLS,TGM2,BICC1,FLNA,TNNT2,COL4A3,MDFIC,LHFPL6,CAVIN2,TNS1,PDLIM2,SHCBP1,SH2D4A,ECT2,TGFB2,ASAP1,GLS,COL4A3,AGFG2'.split(',')
YAP_sig = [x.lower().capitalize() for x in YAP_sig]
YAP_sig = [x for x in YAP_sig if x in lognorm4['FF_YAP5SA_2'].var_names]
YAP_sig = list(set(YAP_sig))

In [ ]:
new_dict['YAP_Signature'] = YAP_sig

## Signatures higher in Running than Mattress at the tumor border (0–50 µm)

In [ ]:
import pandas as pd
import numpy as np

# Initialize list to store results for each sample
results = []

# Process each sample
for sample in sample_list:
    # Get AnnData object for current sample
    bdata = lognorm4[sample]
    
    # Filter for cells in the specified layer
    mask = (bdata.obs['tumor_layer'] == '0um~50um') & (bdata.obs['cell_type'] == 'Melanoma')
    
    # Get signature columns (ending with _Signature)
    signature_cols = [col for col in bdata.obs.columns if col.startswith('aucell_')]
    
    # Calculate mean values for the filtered cells
    means = bdata.obs.loc[mask, signature_cols].mean()
    
    # Get the group value
    group = bdata.obs['group'].iloc[0]
    
    # Create a Series with all required information
    result = pd.Series({
        'group': group,
        **means
    })
    
    # Add to results list with sample as index
    results.append((sample, result))

# Create final DataFrame
final_df = pd.DataFrame([r[1] for r in results], index=[r[0] for r in results])

# Reorder columns to have 'group' first
cols = ['group'] + [col for col in final_df.columns if col != 'group']
final_df = final_df[cols]

In [ ]:
final_df = final_df.dropna(axis = 1)

In [ ]:
final_df

In [ ]:
# Group별 평균 계산
tread_mean = final_df[final_df['group'] == 'Running'].mean()
sponge_mean = final_df[final_df['group'] == 'Mattress'].mean()

# Tread가 Sponge보다 큰 column 찾기
higher_in_tread = tread_mean > sponge_mean

# 결과 필터링 ('group' column 제외)
higher_in_tread = higher_in_tread[1:]  # 첫 번째 'group' column 제외
signature_cols_higher_in_tread = higher_in_tread[higher_in_tread].index.tolist()

# 각 그룹의 평균값과 차이 계산
comparison_df = pd.DataFrame({
    'Running_mean': tread_mean[signature_cols_higher_in_tread],
    'Mattress_mean': sponge_mean[signature_cols_higher_in_tread],
    'Difference': tread_mean[signature_cols_higher_in_tread] - sponge_mean[signature_cols_higher_in_tread]
})

# 차이가 큰 순서대로 정렬
comparison_df = comparison_df.sort_values('Difference', ascending=False)

print(f"Found {len(signature_cols_higher_in_tread)} signatures higher in Running group:")
print(comparison_df)

In [ ]:
comparison_df

## Signatures with a decreasing trend from the border inward (Running group)

In [ ]:
import numpy as np
from scipy import stats

# Layer 순서 정의
layers = ['0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', 
          '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um',
          '500um~550um', '550um~600um', '600um~650um', '650um~700um']

# depth 값을 숫자로 변환 (회귀 분석용)
depth_values = np.arange(len(layers))

# Tread 그룹의 샘플들 찾기
tread_samples = final_df[final_df['group'] == 'Running'].index

# 결과를 저장할 dictionary 초기화
depth_trends = {}
trend_stats = {}

# comparison_df의 각 signature에 대해 분석
for signature in comparison_df.index:
    # 각 layer별 평균값을 저장할 리스트
    layer_means = []
    
    # 각 layer에 대해
    for layer in layers:
        # 모든 Tread 샘플에서 해당 layer의 평균값 계산
        layer_values = []
        for sample in tread_samples:
            adata = lognorm4[sample]
            mask = adata.obs['tumor_layer'] == layer
            mean_value = adata.obs.loc[mask, signature].mean()
            layer_values.append(mean_value)
        
        # 해당 layer의 전체 평균값
        layer_mean = np.mean(layer_values)
        layer_means.append(layer_mean)
    
    # 선형 회귀를 통한 경향성 분석
    slope, intercept, r_value, p_value, std_err = stats.linregress(depth_values, layer_means)
    
    # R-squared 값 계산
    r_squared = r_value ** 2
    
    # 유의한 감소 경향이 있는 경우 저장 (p < 0.05, 음의 기울기)
    if slope < 0 and p_value < 0.05:
        depth_trends[signature] = layer_means
        trend_stats[signature] = {
            'slope': slope,
            'p_value': p_value,
            'r_squared': r_squared
        }

# 결과를 DataFrame으로 변환
decreasing_signatures_df = pd.DataFrame(depth_trends, index=layers).T
stats_df = pd.DataFrame(trend_stats).T

# 통계 결과와 함께 출력
print(f"\nFound {len(depth_trends)} signatures with significant decreasing trend in Running group:")
print("\nStatistical measures:")
print(stats_df.sort_values('slope'))

# 첫 번째 layer (0um~50um)를 기준으로 정규화
normalized_df = decreasing_signatures_df.div(decreasing_signatures_df.iloc[:, 0], axis=0)

# 시각화
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
for signature in normalized_df.index:
    plt.plot(range(len(layers)), normalized_df.loc[signature], 
             label=f"{signature} (R²={trend_stats[signature]['r_squared']:.3f})", 
             marker='o')
plt.xticks(range(len(layers)), layers, rotation=45)
plt.xlabel('Depth')
plt.ylabel('Normalized Expression')
plt.title('Depth-dependent Expression Trends (Normalized to First Layer)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

# 원래 값으로도 시각화
plt.figure(figsize=(15, 6))
for signature in decreasing_signatures_df.index:
    plt.plot(range(len(layers)), decreasing_signatures_df.loc[signature], 
             label=f"{signature} (R²={trend_stats[signature]['r_squared']:.3f})", 
             marker='o')
plt.xticks(range(len(layers)), layers, rotation=45)
plt.xlabel('Depth')
plt.ylabel('Expression Level')
plt.title('Depth-dependent Expression Trends (Original Values)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
decreasing_signatures_df

In [ ]:
i = 0
for signature in decreasing_signatures_df.index[:]:
    layers_of_interest = ['0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um', '500um~550um', '550um~600um', '600um~650um', '650um~700um']

    # Calculate group-level YAP scores
    group_scores_df = calculate_group_signature_scores(adata_dict = lognorm4, layers = layers_of_interest, signature = signature)

    # Plot the results
    plot_group_yap_scores(group_scores_df, signature, layers_of_interest, path = f'./figures/Decrease_{i}_{signature}_image.pdf')
    i += 1

In [ ]:
# sample_list has the same order as combined_adata.obs['sample'].unique() in the original notebook
for term in decreasing_signatures_df.index[:]:
    layers = ['0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', 
          '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um',
          '500um~550um', '550um~600um', '600um~650um', '650um~700um']

    exp_df = pd.DataFrame(index = sample_list, columns=layers)

    exp_df.columns = layers
    exp_df

    for sample in sample_list:
        adata = lognorm4[sample].copy()
        for layer in layers:
            temp = adata[(adata.obs['tumor_layer'] == layer) & (adata.obs['cell_type'] == 'Melanoma')]
            exp_df.loc[sample,layer] = np.mean(temp.obs[term])

    exp_df

    plot_layer_distribution_trend(exp_df, f'{term}')

## Gene-level heatmaps (melanoma pseudospots, 0–150 µm)

In [ ]:
gene_exp = sc.concat(lognorm4)
gene_exp = gene_exp[(gene_exp.obs['cell_type'] == 'Melanoma') & (gene_exp.obs['tumor_layer'].isin(['0um~50um', '50um~100um', '100um~150um']))]

In [ ]:
for sig in decreasing_signatures_df.index[:]:
    sig = sig[len('aucell_'):]
    if sig.startswith('YAP_'):
        sig = 'YAP_Signature'
    plot_gene_group_heatmaps(gene_exp, gene_list=new_dict[sig], group_col='group', term=f'{sig}', genes_per_plot=50)

In [ ]:
for sig in decreasing_signatures_df.index[:]:
    sig = sig[len('aucell_'):]
    if sig.startswith('YAP_'):
        sig = 'YAP_Signature'
    plot_gene_group_heatmaps_raw(gene_exp, gene_list=new_dict[sig], group_col='group', term=f'{sig}', genes_per_plot=50)